In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import random

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

import warnings
warnings.filterwarnings("ignore")

In [2]:
DATA_PATH = 'messy_mashup'

TRAIN_PATH = os.path.join(DATA_PATH,"genres_stems")
TEST_PATH = os.path.join(DATA_PATH,"mashups")
NOISE_PATH = os.path.join(DATA_PATH,"ESC-50-master/audio")

TEST_CSV = os.path.join(DATA_PATH,"test.csv")

In [3]:
count = 0

for root, dirs, files in os.walk(TRAIN_PATH):
    for file in files:
        if file.endswith(".wav"):
            print(file)
            count += 1
            
        if count == 20:
            break
    if count == 20:
        break

bass.wav
drums.wav
other.wav
vocals.wav
bass.wav
drums.wav
other.wav
vocals.wav
bass.wav
drums.wav
other.wav
vocals.wav
bass.wav
drums.wav
other.wav
vocals.wav
bass.wav
drums.wav
other.wav
vocals.wav


In [4]:
wav_count = 0

for root, dirs, files in os.walk(TRAIN_PATH):
    for file in files:
        if file.endswith(".wav"):
            wav_count += 1

print("Total wav files:", wav_count)

Total wav files: 4000


In [5]:
print(TRAIN_PATH)
print(os.path.exists(TRAIN_PATH))
print(os.listdir(TRAIN_PATH))

messy_mashup\genres_stems
True
['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


In [6]:
genre_path = os.path.join(TRAIN_PATH,"blues")

print(genre_path)
print(os.path.exists(genre_path))
print(os.listdir(genre_path)[:5])

messy_mashup\genres_stems\blues
True
['blues.00000', 'blues.00001', 'blues.00002', 'blues.00003', 'blues.00004']


In [7]:
song_folder = os.listdir(genre_path)[0]

song_path = os.path.join(genre_path,song_folder)

print(song_path)
print(os.listdir(song_path))

messy_mashup\genres_stems\blues\blues.00000
['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']


In [8]:
stems = [
    os.path.join(song_path,"drums.wav"),
    os.path.join(song_path,"bass.wav"),
    os.path.join(song_path,"vocals.wav"),
    os.path.join(song_path,"other.wav")
]

for s in stems:
    print(s, os.path.exists(s))

messy_mashup\genres_stems\blues\blues.00000\drums.wav True
messy_mashup\genres_stems\blues\blues.00000\bass.wav True
messy_mashup\genres_stems\blues\blues.00000\vocals.wav True
messy_mashup\genres_stems\blues\blues.00000\other.wav True


In [9]:
import librosa

y, sr = librosa.load(stems[0], sr=22050)

print(len(y), sr)

661794 22050


In [10]:
import librosa
import numpy as np

def extract_mel(y, sr=22050):

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=128
    )

    mel = librosa.power_to_db(mel)

    # keep fixed width
    if mel.shape[1] > 512:
        mel = mel[:, :512]
    else:
        mel = np.pad(mel, ((0,0),(0,512-mel.shape[1])))

    return mel

In [11]:
mel = extract_mel(y)

print(mel.shape)

(128, 512)


In [12]:
import random
import librosa

def tempo_shift(y):

    rate = random.uniform(0.9, 1.1)

    try:
        y = librosa.effects.time_stretch(y, rate)
    except:
        pass

    return y

In [13]:
def add_noise(y):

    noise = np.random.randn(len(y))

    y = y + 0.003 * noise

    return y

In [14]:
def create_mashup(stems):

    tracks = []

    for stem in stems:

        y, sr = librosa.load(stem, sr=22050)

        y = tempo_shift(y)

        tracks.append(y)

    min_len = min(len(t) for t in tracks)

    tracks = [t[:min_len] for t in tracks]

    mix = sum(tracks)

    mix = mix / np.max(np.abs(mix))

    mix = add_noise(mix)

    return mix

In [15]:
mix = create_mashup(stems)

print(len(mix))

661794


In [16]:
X = []
y = []

for genre in os.listdir(TRAIN_PATH):

    genre_path = os.path.join(TRAIN_PATH, genre)

    if not os.path.isdir(genre_path):
        continue

    for song in os.listdir(genre_path):

        song_path = os.path.join(genre_path, song)

        if not os.path.isdir(song_path):
            continue

        stems = [
            os.path.join(song_path, "drums.wav"),
            os.path.join(song_path, "bass.wav"),
            os.path.join(song_path, "vocals.wav"),
            os.path.join(song_path, "other.wav")
        ]

        # ensure all stems exist
        if not all(os.path.exists(s) for s in stems):
            continue

        try:

            mashup = create_mashup(stems)

            mel = extract_mel(mashup)

            X.append(mel)

            y.append(genre)

        except Exception as e:

            print("Error:", e)

            continue


print("Samples loaded:", len(X))

Samples loaded: 1000


In [17]:
#Genre Labels

genres = [
"blues","classical","country","disco","hiphop",
"jazz","metal","pop","reggae","rock"
]

encoder = LabelEncoder()
encoder.fit(genres)

LabelEncoder()

In [18]:
#Load Noise Files

noise_files = []

for root,dirs,files in os.walk(NOISE_PATH):
    
    for file in files:
        
        if file.endswith(".wav"):
            
            noise_files.append(os.path.join(root,file))

print("Noise samples:",len(noise_files))

Noise samples: 2000


In [19]:
#Audio Utilities

def add_noise(y):

    noise_file = random.choice(noise_files)

    noise, sr = librosa.load(noise_file, sr=22050)

    if len(noise) < len(y):
        repeat = int(np.ceil(len(y)/len(noise)))
        noise = np.tile(noise, repeat)

    noise = noise[:len(y)]

    noise = noise * random.uniform(0.01,0.05)

    return y + noise

In [20]:
#Create Mashup

def create_mashup(stems):

    tracks = []

    for stem in stems:

        y, sr = librosa.load(stem, sr=22050)

        y = tempo_shift(y)

        tracks.append(y)

    min_len = min(len(t) for t in tracks)

    tracks = [t[:min_len] for t in tracks]

    mix = sum(tracks)

    mix = mix / np.max(np.abs(mix))

    mix = add_noise(mix)

    return mix

In [21]:
#Mel Spectrogram Feature

def extract_mel(y):

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=22050,
        n_mels=128
    )

    mel = librosa.power_to_db(mel)

    mel = mel[:,:512]

    return mel

In [22]:
#Convert to Arrays

X = np.array(X)
y = np.array(y)

X = X[...,np.newaxis]

y = encoder.transform(y)

print(X.shape)

(1000, 128, 512, 1)


In [23]:
#Train Validation Split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [24]:
#CNN Model

model = Sequential([

layers.Conv2D(32,(3,3),activation="relu",input_shape=X_train.shape[1:]),
layers.BatchNormalization(),
layers.MaxPooling2D(),

layers.Conv2D(64,(3,3),activation="relu"),
layers.BatchNormalization(),
layers.MaxPooling2D(),

layers.Conv2D(128,(3,3),activation="relu"),
layers.BatchNormalization(),
layers.MaxPooling2D(),

layers.Flatten(),

layers.Dense(256,activation="relu"),
layers.Dropout(0.3),

layers.Dense(10,activation="softmax")

])

In [25]:
#Compile model

model.compile(
optimizer="adam",
loss="sparse_categorical_crossentropy",
metrics=["accuracy"]
)

In [26]:
#train model

history = model.fit(

X_train,
y_train,

epochs=20,
batch_size=32,

validation_data=(X_val,y_val)

)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 182s 5s/step - accuracy: 0.1824 - loss: 53.7276 - val_accuracy: 0.1100 - val_loss: 123.9077
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 231s 9s/step - accuracy: 0.2643 - loss: 3.7614 - val_accuracy: 0.1150 - val_loss: 15.8439
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 86s 3s/step - accuracy: 0.2218 - loss: 2.1455 - val_accuracy: 0.1150 - val_loss: 7.7931
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 106s 4s/step - accuracy: 0.1595 - loss: 2.1767 - val_accuracy: 0.1200 - val_loss: 5.8800
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 74s 3s/step - accuracy: 0.2095 - loss: 2.1861 - val_accuracy: 0.1500 - val_loss: 2.7876
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 89s 3s/step - accuracy: 0.2555 - loss: 2.0838 - val_accuracy: 0.1850 - val_loss: 2.6342
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 80s 3s/step - accuracy: 0.2291 - loss: 2.0623 - val_accuracy: 0.2050 - val_loss: 2.5031
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 64s 3s/step - accuracy: 0.2299 - loss: 2.0242 - val_accuracy: 0.2050 - va

In [27]:
#Validation Macro F1

pred = model.predict(X_val)

pred_labels = np.argmax(pred,axis=1)

f1 = f1_score(y_val,pred_labels,average="macro")

print("Macro F1:",f1)

7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step 
Macro F1: 0.34718219333600453


In [32]:
print(TEST_PATH)
print(os.path.exists(TEST_PATH))
print(os.listdir(TEST_PATH)[:10])

messy_mashup\mashups
True
['song0001.wav', 'song0002.wav', 'song0003.wav', 'song0004.wav', 'song0005.wav', 'song0006.wav', 'song0007.wav', 'song0008.wav', 'song0009.wav', 'song0010.wav']


In [34]:
def extract_mel(y, sr=22050, max_len=512):

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=128
    )

    mel = librosa.power_to_db(mel)

    # fix width
    if mel.shape[1] > max_len:
        mel = mel[:, :max_len]

    else:
        pad_width = max_len - mel.shape[1]
        mel = np.pad(mel, ((0,0),(0,pad_width)), mode='constant')

    return mel

In [35]:
test_df = pd.read_csv(TEST_CSV)

X_test = []
ids = []

for file in test_df["id"]:

    path = os.path.join(TEST_PATH, f"song{file:04d}.wav")

    y_audio, sr = librosa.load(path, sr=22050)

    mel = extract_mel(y_audio)

    X_test.append(mel)

    ids.append(file)

X_test = np.array(X_test)

X_test = X_test[..., np.newaxis]

In [36]:
#Predict test

pred = model.predict(X_test)

pred_labels = np.argmax(pred,axis=1)

genres_pred = encoder.inverse_transform(pred_labels)

95/95 ━━━━━━━━━━━━━━━━━━━━ 47s 418ms/step


In [37]:
#Create Submission

submission = pd.DataFrame({

"id":ids,
"genre":genres_pred

})

submission.to_csv("submission.csv",index=False)

submission.head()

,id,genre
0,1,disco
1,2,jazz
2,3,country
3,4,rock
4,5,country
